In [76]:
# Library yang digunakan
from nltk.tokenize import word_tokenize
import string
import pandas as pd
import numpy as np
import nltk
import emoji
import re
from sklearn.pipeline import Pipeline
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mirur\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

## 💡 Load Data

In [77]:
#Load Dataser
version = '5.5'
file_path = f"../data/review_genshin_{version}_cleaned.csv"
data = pd.read_csv(file_path)

In [78]:
# Show Dataset
data.head(10)

,ReviewID,Content,Score,ThumbsUpCount,at,AppVersion
0,2cf85c2d-bc47-4030-9eca-ac000d46d82f,HELLO GENSHIN IMPACT I HAVE A LITTLE COMPLAINT...,5,0,2025-04-10 18:30:38,5.5
1,c97ec484-677d-4d17-b330-39502dd9b045,"Saran buat dev dioptimalkan lagi game ya, 2020...",1,0,2025-04-09 20:08:42,5.5
2,4550d4ff-d199-4983-81f5-a299b923ab19,good,5,0,2025-04-09 19:57:42,5.5
3,4318f7e7-e90b-4fd8-acae-45a2dd6e7452,game nya asik bet anjay,5,0,2025-04-09 19:55:05,5.5
4,fa38b679-0c26-4922-82c5-08474ed35fed,Ini game atau seni yaallah soalnya bgus bgt!!!...,5,0,2025-04-09 19:46:49,5.5
5,0c916f46-3643-41fe-a684-fa4f9a2e3dad,good game,5,0,2025-04-09 19:32:24,5.5
6,65b278f4-fb78-477b-8368-dd5f310f6d27,oke saya tidak ada lagi yang ingin di sampaika...,5,211,2025-04-09 19:26:42,5.5
7,5708cf2e-fe86-4db2-bba1-2da54f464898,game ny oke tp g tau np itu skrg klo mau downl...,3,0,2025-04-09 19:10:07,5.5
8,ad36239f-9f4d-4cb7-a8cd-57a68a7e1466,perfect.,5,0,2025-04-09 19:03:40,5.5
9,70e0d324-08bb-4e23-9690-01540ce3f137,"Bagus sih, untung gak kikir seperti dulu wkwk",5,0,2025-04-09 18:59:10,5.5


In [79]:
data = data[data['Score'].between(1, 3)]

In [80]:
# proses case folding 
def casefolding(Content):
    Content = Content.lower()
    return Content
data['Content'] = data['Content'].apply(casefolding)
data.head()

,ReviewID,Content,Score,ThumbsUpCount,at,AppVersion
1,c97ec484-677d-4d17-b330-39502dd9b045,"saran buat dev dioptimalkan lagi game ya, 2020...",1,0,2025-04-09 20:08:42,5.5
7,5708cf2e-fe86-4db2-bba1-2da54f464898,game ny oke tp g tau np itu skrg klo mau downl...,3,0,2025-04-09 19:10:07,5.5
18,10d4af53-7931-46d0-b46e-918ad1034af7,skrng berasa jadi game bayi,1,0,2025-04-09 16:03:06,5.5
21,4ff06131-445f-42dc-97b9-6a5534edf433,"game ini makin lama makin ampas, jaringan, gac...",1,5,2025-04-09 15:01:27,5.5
26,b5995cea-8885-462c-af95-1e732b72744a,ukuran datanya gak ngotak sumpah download haru...,1,0,2025-04-09 12:53:25,5.5


In [81]:
def cleansing(Content):
    Content = emoji.replace_emoji(Content, replace='')  # Hapus semua emoji
    Content = Content.strip(" ")
    Content = re.sub(r'[?|$|.|!_:")(-+,]', '', Content)
    Content = re.sub(r'\d+', '', Content)
    Content = re.sub(r"\b[a-zA-Z]\b", "", Content)
    Content = re.sub('\s+', ' ', Content)
    return Content

data['Content'] = data['Content'].apply(cleansing)

<>:7: SyntaxWarning: invalid escape sequence '\s'
<>:7: SyntaxWarning: invalid escape sequence '\s'
C:\Users\mirur\AppData\Local\Temp\ipykernel_1792\3986272389.py:7: SyntaxWarning: invalid escape sequence '\s'
  Content = re.sub('\s+', ' ', Content)


In [82]:
# NLTK word tokenize


def word_tokenize_wrapper(text):
    return word_tokenize(text)


data['Content'] = data['Content'].apply(word_tokenize_wrapper)
data.head()

,ReviewID,Content,Score,ThumbsUpCount,at,AppVersion
1,c97ec484-677d-4d17-b330-39502dd9b045,"[saran, buat, dev, dioptimalkan, lagi, game, y...",1,0,2025-04-09 20:08:42,5.5
7,5708cf2e-fe86-4db2-bba1-2da54f464898,"[game, ny, oke, tp, tau, np, itu, skrg, klo, m...",3,0,2025-04-09 19:10:07,5.5
18,10d4af53-7931-46d0-b46e-918ad1034af7,"[skrng, berasa, jadi, game, bayi]",1,0,2025-04-09 16:03:06,5.5
21,4ff06131-445f-42dc-97b9-6a5534edf433,"[game, ini, makin, lama, makin, ampas, jaringa...",1,5,2025-04-09 15:01:27,5.5
26,b5995cea-8885-462c-af95-1e732b72744a,"[ukuran, datanya, gak, ngotak, sumpah, downloa...",1,0,2025-04-09 12:53:25,5.5


In [83]:
normalizad_word = pd.read_csv("../data/normalisasi.csv")

normalizad_word_dict = {}

for index, row in normalizad_word.iterrows():
    if row[0] not in normalizad_word_dict:
        normalizad_word_dict[row[0]] = row[1] 

def normalized_term(document):
    return [normalizad_word_dict[term] if term in normalizad_word_dict else term for term in document]

data['Content'] = data['Content'].apply(normalized_term)

data.head()

C:\Users\mirur\AppData\Local\Temp\ipykernel_1792\2302089821.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if row[0] not in normalizad_word_dict:
C:\Users\mirur\AppData\Local\Temp\ipykernel_1792\2302089821.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  normalizad_word_dict[row[0]] = row[1]


,ReviewID,Content,Score,ThumbsUpCount,at,AppVersion
1,c97ec484-677d-4d17-b330-39502dd9b045,"[saran, buat, dev, dioptimalkan, lagi, game, y...",1,0,2025-04-09 20:08:42,5.5
7,5708cf2e-fe86-4db2-bba1-2da54f464898,"[game, nya, oke, tetapi, tahu, np, itu, sekara...",3,0,2025-04-09 19:10:07,5.5
18,10d4af53-7931-46d0-b46e-918ad1034af7,"[sekarang, berasa, jadi, game, bayi]",1,0,2025-04-09 16:03:06,5.5
21,4ff06131-445f-42dc-97b9-6a5534edf433,"[game, ini, makin, lama, makin, ampas, jaringa...",1,5,2025-04-09 15:01:27,5.5
26,b5995cea-8885-462c-af95-1e732b72744a,"[ukuran, datanya, tidak, ngotak, sumpah, downl...",1,0,2025-04-09 12:53:25,5.5


In [84]:

from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')
sw = pd.read_csv("../data/stopwords_id.csv")


def stopword_removal(Content):
    filtering = stopwords.words('indonesian', 'english')
    filtering.extend(sw)
    x = []
    data = []

    def myFunc(x):
        if x in filtering:
            return False
        else:
            return True
    fit = filter(myFunc, Content)
    for x in fit:
        data.append(x)
    return data


data['Content'] = data['Content'].apply(stopword_removal)
data.head()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mirur\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,ReviewID,Content,Score,ThumbsUpCount,at,AppVersion
1,c97ec484-677d-4d17-b330-39502dd9b045,"[saran, dev, dioptimalkan, game, ya, edit, ula...",1,0,2025-04-09 20:08:42,5.5
7,5708cf2e-fe86-4db2-bba1-2da54f464898,"[game, nya, oke, np, download, data, aneh, ora...",3,0,2025-04-09 19:10:07,5.5
18,10d4af53-7931-46d0-b46e-918ad1034af7,"[berasa, game, bayi]",1,0,2025-04-09 16:03:06,5.5
21,4ff06131-445f-42dc-97b9-6a5534edf433,"[game, ampas, jaringan, gacha, karakter, artef...",1,5,2025-04-09 15:01:27,5.5
26,b5995cea-8885-462c-af95-1e732b72744a,"[ukuran, datanya, ngotak, sumpah, download, nu...",1,0,2025-04-09 12:53:25,5.5


### Proses Stemming dan Membuat file data baru (dataset yang sudah dibersihkan melalui proses NLTK)

In [85]:
# proses stemming
def stemming(Content):
    factory = StemmerFactory()
    stemmer = factory.create_stemmer()
    do = []
    for w in Content:
        dt = stemmer.stem(w)
        do.append(dt)
    d_clean = []
    d_clean = " ".join(do)
    print(d_clean)
    return d_clean


data['Content'] = data['Content'].apply(stemming)

data.to_csv(f'../data/review_genshin_{version}_clear.csv', index=False)
data_clean = pd.read_csv(
    f'../data/review_genshin_{version}_clear.csv', encoding='latin1')
data_clean.head()

saran dev optimal game ya edit ulas pv buset dah story nya sangka game kasih bintang bagus lore storynya pingin ngikuti game edit maaf lorenya kek game bayi ya natlan negara perang kebun binatang kasih bintang kecewa benerrrrr
game nya oke np download data aneh orang selesai download data pas masuk game nya mlh selesai nya mah jdinya makan storage sm pulsa tolong dibenerin hoyo
asa game bayi
game ampas jaring gacha karakter artefak kit combat story ampas game ramah fp habis bosan pensi bosen
ukur data ngotak sumpah download nunggu jam jam hp panas bet
apa game fitur skip lagigame nya makan game nya pelit
woi hoyo
cakap skip buang reward tara kerja quest reward gacha player fp
game hoyoverse game sapi perah doang rate ampas
isi suara en hilang alas tanda tanda fix
optimal sinyal busuk banget mobile disconnect menit pdhl main game aman lancarr
seru
skip button just yapping game
samachar cowo nya dikit banget laut deh
game kikir main hp bagus ngelag explorenya rada gila mula suruh explore

,ReviewID,Content,Score,ThumbsUpCount,at,AppVersion
0,c97ec484-677d-4d17-b330-39502dd9b045,saran dev optimal game ya edit ulas pv buset d...,1,0,2025-04-09 20:08:42,5.5
1,5708cf2e-fe86-4db2-bba1-2da54f464898,game nya oke np download data aneh orang seles...,3,0,2025-04-09 19:10:07,5.5
2,10d4af53-7931-46d0-b46e-918ad1034af7,asa game bayi,1,0,2025-04-09 16:03:06,5.5
3,4ff06131-445f-42dc-97b9-6a5534edf433,game ampas jaring gacha karakter artefak kit c...,1,5,2025-04-09 15:01:27,5.5
4,b5995cea-8885-462c-af95-1e732b72744a,ukur data ngotak sumpah download nunggu jam ja...,1,0,2025-04-09 12:53:25,5.5
